# Molecular-property overlap audit for `MERGED_DATASET`

This notebook downloads and audits molecular-property datasets against `datasets_old/merged/merged_train_val_all.parquet`.

It includes:

- **TDC ADMET Group**: the benchmark endpoints with official `train_val` and held-out `test` CSVs.
- **MoleculeNet complement**: ESOL, AstraZeneca **logD at pH 7.4**, BBBP, ClinTox, SIDER, and Tox21 raw releases. These are reported as full releases because this notebook does not impose a new split on them.
- **PAMPA_NCATS**: the TDC NCATS passive-permeability dataset, downloaded directly from the TDC Harvard Dataverse release and audited as a full release.
- **SangsterLogP**: the 2026 curated experimental logP dataset from Zenodo, filtered to `Ionisation-based class = P` before auditing as a full release using the `logP` target.

The primary molecular match is the **full InChIKey** derived with RDKit. Canonical isomeric SMILES are recorded for inspection, but are not the sole join key. The report also counts connectivity-only matches (the first InChIKey block); these are **candidates for manual review**, not automatic label joins, because they can differ in stereochemistry, salt form, tautomerism, or protonation.

The notebook never rewrites `merged_train_val_all.parquet`. It writes downloaded property releases under `datasets_old/properties/` and audit outputs under `datasets_old/properties/admet_overlap/`.


## Requirements

Run with the repository's main NMR environment, which must provide:

- `pandas`
- `pyarrow`
- `RDKit`
- `openpyxl` (only needed to read the SangsterLogP XLSX file)

Downloads use only Python's standard library.

Primary sources:

- [TDC ADMET benchmark](https://tdcommons.ai/benchmark/admet_group/overview/)
- [TDC PAMPA_NCATS](https://tdcommons.ai/single_pred_tasks/adme/)
- [MoleculeNet](https://pubs.rsc.org/en/content/articlehtml/2018/sc/c7sc02664a) — the Lipophilicity release is experimental octanol/water **logD at pH 7.4** for 4,200 compounds.
- [SangsterLogP](https://zenodo.org/records/19387552) — curated experimental/adjusted logP values for 23,520 unique compounds.


In [2]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import requests
from zipfile import ZipFile

import pandas as pd
import pyarrow.parquet as pq
from rdkit import Chem, RDLogger

# Invalid source structures are counted explicitly below. Silence RDKit parser
# diagnostics so a few malformed records do not flood notebook output.
RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'datasets_old').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

MERGED_DATASET_PATH = REPO_ROOT / 'datasets/intermediate/train_val_filtered.parquet'
PROPERTY_ROOT = REPO_ROOT / 'datasets_old/properties'
TDC_ROOT = PROPERTY_ROOT / 'tdc_admet'
TDC_ARCHIVE = TDC_ROOT / 'admet_group.zip'
TDC_GROUP_ROOT = TDC_ROOT / 'admet_group'
TDC_EXTRA_ROOT = PROPERTY_ROOT / 'tdc_extra'
MOLECULENET_ROOT = PROPERTY_ROOT / 'moleculenet_admet'
SANGSTER_ROOT = PROPERTY_ROOT / 'sangster_logp'
REPORT_ROOT = PROPERTY_ROOT / 'admet_overlap'

# Set to True only when merged_train_val_all.parquet has changed and the cached index
# needs rebuilding. The default reuses a cache only when source size and mtime match.
FORCE_REBUILD_MERGED_INDEX = True

TDC_ADMET_URL = 'https://dataverse.harvard.edu/api/access/datafile/4426004'

# PAMPA_NCATS is a TDC ADME dataset but is not part of the historical ADMET benchmark
# archive above. TDC currently maps pampa_ncats to Harvard Dataverse datafile 6695858.
TDC_EXTRA_FILES = {
    'pampa_ncats': {
        'filename': 'pampa_ncats.tab',
        'url': 'https://dataverse.harvard.edu/api/access/datafile/6695858',
        'reader': 'tsv',
        'smiles_column': 'Drug',
        'label_column': 'Y',
    },
}

MOLECULENET_FILES = {
    'esol': {
        'filename': 'delaney-processed.csv',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv',
        'smiles_column': 'smiles',
    },
    # MoleculeNet calls this dataset "Lipophilicity", but the experimental endpoint
    # is octanol/water distribution coefficient (logD) measured at pH 7.4.
    'logd74_astrazeneca': {
        'filename': 'Lipophilicity.csv',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv',
        'smiles_column': 'smiles',
        'label_column': 'exp',
    },
    'bbbp': {
        'filename': 'BBBP.csv',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv',
        'smiles_column': 'smiles',
    },
    'clintox': {
        'filename': 'clintox.csv.gz',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/clintox.csv.gz',
        'smiles_column': 'smiles',
    },
    'sider': {
        'filename': 'sider.csv.gz',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/sider.csv.gz',
        'smiles_column': 'smiles',
    },
    'tox21': {
        'filename': 'tox21.csv.gz',
        'url': 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz',
        'smiles_column': 'smiles',
    },
}

SANGSTER_FILES = {
    'sangster_logp': {
        'filename': 'Datasets.xlsx',
        'url': 'https://zenodo.org/records/19387552/files/Datasets.xlsx?download=1',
        'reader': 'excel_auto',
        'smiles_column': 'SMILES',
        'label_column': 'logP',
        'filters': {'Ionisation-based class': 'P'},
    },
}


def download_if_missing(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0:
        print(f'Reusing {destination.relative_to(REPO_ROOT)}')
        return

    print(f'Downloading {url}')

    temporary = destination.with_suffix(destination.suffix + '.part')
    temporary.unlink(missing_ok=True)

    headers = {
        'User-Agent': 'Mozilla/5.0 (compatible; NMR-overlap-audit/1.0)',
        'Accept': '*/*',
    }

    try:
        with requests.get(
            url,
            stream=True,
            timeout=120,
            headers=headers,
            allow_redirects=True,
        ) as response:
            response.raise_for_status()

            with temporary.open('wb') as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)

        temporary.replace(destination)

    except Exception:
        temporary.unlink(missing_ok=True)
        raise


def ensure_property_sources() -> None:
    if not TDC_GROUP_ROOT.is_dir():
        download_if_missing(TDC_ADMET_URL, TDC_ARCHIVE)
        print('Extracting the TDC ADMET archive')
        with ZipFile(TDC_ARCHIVE) as archive:
            archive.extractall(TDC_ROOT)

    for info in MOLECULENET_FILES.values():
        download_if_missing(info['url'], MOLECULENET_ROOT / info['filename'])

    for info in TDC_EXTRA_FILES.values():
        download_if_missing(info['url'], TDC_EXTRA_ROOT / info['filename'])

    for info in SANGSTER_FILES.values():
        download_if_missing(info['url'], SANGSTER_ROOT / info['filename'])


ensure_property_sources()

if not MERGED_DATASET_PATH.is_file():
    raise FileNotFoundError(f'Missing {MERGED_DATASET_PATH}')

print(f'MERGED_DATASET: {MERGED_DATASET_PATH}')
print(f'TDC benchmark endpoints found: {len(list(TDC_GROUP_ROOT.glob("*/test.csv")))}')
print(f'MoleculeNet releases found: {len(MOLECULENET_FILES)}')
print(f'Extra TDC releases found: {len(TDC_EXTRA_FILES)}')
print(f'Sangster releases found: {len(SANGSTER_FILES)}')


Reusing datasets_old/properties/moleculenet_admet/delaney-processed.csv
Reusing datasets_old/properties/moleculenet_admet/Lipophilicity.csv
Reusing datasets_old/properties/moleculenet_admet/BBBP.csv
Reusing datasets_old/properties/moleculenet_admet/clintox.csv.gz
Reusing datasets_old/properties/moleculenet_admet/sider.csv.gz
Reusing datasets_old/properties/moleculenet_admet/tox21.csv.gz
Reusing datasets_old/properties/tdc_extra/pampa_ncats.tab
Reusing datasets_old/properties/sangster_logp/Datasets.xlsx
MERGED_DATASET: /mnt/filesystem-test/nmr/datasets/intermediate/train_val_filtered.parquet
TDC benchmark endpoints found: 22
MoleculeNet releases found: 6
Extra TDC releases found: 1
Sangster releases found: 1


In [3]:
INDEX_CACHE = REPORT_ROOT / 'MERGED_DATASET_identity_index.parquet'
INDEX_METADATA = REPORT_ROOT / 'MERGED_DATASET_identity_index_metadata.json'


def structure_identifiers(smiles: object) -> dict[str, str | None]:
    if not isinstance(smiles, str) or not smiles.strip():
        return {
            'canonical_smiles_rdkit': None,
            'full_inchikey': None,
            'connectivity_inchikey': None,
        }

    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        return {
            'canonical_smiles_rdkit': None,
            'full_inchikey': None,
            'connectivity_inchikey': None,
        }

    canonical_smiles = Chem.MolToSmiles(
        molecule,
        canonical=True,
        isomericSmiles=True,
    )

    try:
        full_inchikey = Chem.MolToInchiKey(molecule)
    except Exception:
        full_inchikey = None

    connectivity_inchikey = (
        full_inchikey.split('-')[0]
        if full_inchikey
        else None
    )

    return {
        'canonical_smiles_rdkit': canonical_smiles,
        'full_inchikey': full_inchikey,
        'connectivity_inchikey': connectivity_inchikey,
    }


def merged_source_state(path: Path) -> dict[str, int]:
    stat = path.stat()
    return {
        'size_bytes': stat.st_size,
        'mtime_ns': stat.st_mtime_ns,
    }


def build_merged_identity_index(path: Path) -> tuple[pd.DataFrame, dict[str, int]]:
    parquet_file = pq.ParquetFile(path)
    if 'smiles_canonical' not in parquet_file.schema_arrow.names:
        raise KeyError('MERGED_DATASET has no smiles_canonical column')

    by_full_inchikey = {}
    invalid_or_unkeyed_rows = 0
    processed_rows = 0

    for batch in parquet_file.iter_batches(
        batch_size=10_000,
        columns=['smiles_canonical'],
    ):
        for smiles in batch.column(0).to_pylist():
            identifiers = structure_identifiers(smiles)
            full_inchikey = identifiers['full_inchikey']

            if full_inchikey is None:
                invalid_or_unkeyed_rows += 1
                continue

            row = by_full_inchikey.setdefault(
                full_inchikey,
                {
                    'full_inchikey': full_inchikey,
                    'connectivity_inchikey': identifiers['connectivity_inchikey'],
                    'nmr_smiles_canonical': identifiers['canonical_smiles_rdkit'],
                    'nmr_record_count': 0,
                },
            )
            row['nmr_record_count'] += 1

        processed_rows += batch.num_rows
        if processed_rows % 50_000 == 0:
            print(f'Processed {processed_rows:,} MERGED_DATASET records')

    index = pd.DataFrame(by_full_inchikey.values())
    summary = {
        'nmr_records': processed_rows,
        'nmr_invalid_or_unkeyed_rows': invalid_or_unkeyed_rows,
        'nmr_unique_molecules': len(index),
        'nmr_duplicate_records_beyond_unique_molecules': processed_rows - invalid_or_unkeyed_rows - len(index),
    }
    return index, summary


def load_or_build_merged_index() -> tuple[pd.DataFrame, dict[str, int]]:
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    state = merged_source_state(MERGED_DATASET_PATH)

    if not FORCE_REBUILD_MERGED_INDEX and INDEX_CACHE.exists() and INDEX_METADATA.exists():
        cached_metadata = json.loads(INDEX_METADATA.read_text())
        if cached_metadata.get('merged_source_state') == state:
            print('Reusing cached MERGED_DATASET identity index')
            return pd.read_parquet(INDEX_CACHE), cached_metadata['summary']

    print('Building MERGED_DATASET identity index from Parquet batches')
    index, summary = build_merged_identity_index(MERGED_DATASET_PATH)
    index.to_parquet(INDEX_CACHE, index=False)
    INDEX_METADATA.write_text(json.dumps({
        'merged_source_state': state,
        'summary': summary,
    }, indent=2))
    return index, summary


nmr_index, nmr_summary = load_or_build_merged_index()
print(json.dumps(nmr_summary, indent=2))

Building MERGED_DATASET identity index from Parquet batches
Processed 50,000 MERGED_DATASET records
Processed 100,000 MERGED_DATASET records
Processed 150,000 MERGED_DATASET records
Processed 200,000 MERGED_DATASET records
Processed 250,000 MERGED_DATASET records
Processed 300,000 MERGED_DATASET records
Processed 350,000 MERGED_DATASET records
Processed 400,000 MERGED_DATASET records
Processed 450,000 MERGED_DATASET records
Processed 500,000 MERGED_DATASET records
Processed 550,000 MERGED_DATASET records
Processed 600,000 MERGED_DATASET records
Processed 650,000 MERGED_DATASET records
Processed 700,000 MERGED_DATASET records
Processed 750,000 MERGED_DATASET records
Processed 800,000 MERGED_DATASET records
Processed 850,000 MERGED_DATASET records
Processed 900,000 MERGED_DATASET records
Processed 950,000 MERGED_DATASET records
Processed 1,000,000 MERGED_DATASET records
Processed 1,050,000 MERGED_DATASET records
Processed 1,100,000 MERGED_DATASET records
Processed 1,150,000 MERGED_DATASE

In [4]:
def annotate_property_structures(frame: pd.DataFrame, smiles_column: str) -> pd.DataFrame:
    result = frame.copy().reset_index(drop=True)
    result['_source_row'] = range(len(result))
    identifiers = [structure_identifiers(value) for value in result[smiles_column]]
    return pd.concat([result, pd.DataFrame(identifiers)], axis=1)


def read_property_frame(release: dict[str, object]) -> pd.DataFrame:
    path = Path(release['path'])
    reader = release.get('reader', 'csv')

    if reader == 'csv':
        return pd.read_csv(path)

    if reader == 'tsv':
        return pd.read_csv(path, sep='\t')

    if reader == 'excel_auto':
        # SangsterLogP ships as a workbook containing the main curated dataset plus
        # smaller validation/test sheets. Select the largest sheet that contains the
        # required structure and label columns, rather than depending on a sheet name.
        sheets = pd.read_excel(path, sheet_name=None)
        smiles_column = release['smiles_column']
        label_column = release.get('label_column')

        candidates = []
        for sheet_name, frame in sheets.items():
            has_smiles = smiles_column in frame.columns
            has_label = label_column is None or label_column in frame.columns
            if has_smiles and has_label:
                candidates.append((sheet_name, frame))

        if not candidates:
            available = {name: list(frame.columns) for name, frame in sheets.items()}
            raise KeyError(
                f'Could not find a worksheet in {path} with required columns '
                f'{smiles_column!r} and {label_column!r}. Available columns: {available}'
            )

        sheet_name, frame = max(candidates, key=lambda item: len(item[1]))
        frame = frame.copy()
        frame['_source_sheet'] = sheet_name
        print(f'Selected SangsterLogP worksheet {sheet_name!r} with {len(frame):,} rows')
        return frame

    raise ValueError(f'Unknown reader {reader!r} for {path}')


def collect_property_releases() -> list[dict[str, object]]:
    releases = []

    for test_path in sorted(TDC_GROUP_ROOT.glob('*/test.csv')):
        endpoint = test_path.parent.name
        for split_name, filename in [('test', 'test.csv'), ('train_val', 'train_val.csv')]:
            path = test_path.parent / filename
            releases.append({
                'source': 'TDC ADMET',
                'dataset': endpoint,
                'split': split_name,
                'path': path,
                'reader': 'csv',
                'smiles_column': 'Drug',
                'label_column': 'Y',
            })

    for dataset, info in MOLECULENET_FILES.items():
        releases.append({
            'source': 'MoleculeNet',
            'dataset': dataset,
            'split': 'full_release_no_split',
            'path': MOLECULENET_ROOT / info['filename'],
            'reader': 'csv',
            'smiles_column': info['smiles_column'],
            'label_column': info.get('label_column'),
            'filters': info.get('filters', {}),
        })

    for dataset, info in TDC_EXTRA_FILES.items():
        releases.append({
            'source': 'TDC ADME full release',
            'dataset': dataset,
            'split': 'full_release_no_split',
            'path': TDC_EXTRA_ROOT / info['filename'],
            'reader': info['reader'],
            'smiles_column': info['smiles_column'],
            'label_column': info.get('label_column'),
            'filters': info.get('filters', {}),
        })

    for dataset, info in SANGSTER_FILES.items():
        releases.append({
            'source': 'SangsterLogP',
            'dataset': dataset,
            'split': 'full_release_no_split',
            'path': SANGSTER_ROOT / info['filename'],
            'reader': info['reader'],
            'smiles_column': info['smiles_column'],
            'label_column': info.get('label_column'),
            'filters': info.get('filters', {}),
        })

    return releases


def audit_release(
    release: dict[str, object],
    merged_index: pd.DataFrame,
) -> tuple[dict[str, object], pd.DataFrame]:
    frame = read_property_frame(release)
    for column, value in release.get('filters', {}).items():
        if column not in frame.columns:
            raise KeyError(f'{release["path"]} has no filter column {column!r}')
        rows_before_filter = len(frame)
        frame = frame.loc[frame[column].eq(value)].copy()
        print(
            f'Filtered {release["dataset"]} on {column!r} == {value!r}: '
            f'{rows_before_filter:,} -> {len(frame):,} rows'
        )
    smiles_column = release['smiles_column']
    if smiles_column not in frame.columns:
        raise KeyError(f'{release["path"]} has no {smiles_column!r} column')

    label_column = release.get('label_column')
    if label_column is not None and label_column not in frame.columns:
        raise KeyError(f'{release["path"]} has no expected label column {label_column!r}')

    properties = annotate_property_structures(frame, smiles_column)
    nmr_full = set(merged_index['full_inchikey'].dropna())
    nmr_connectivity = set(merged_index['connectivity_inchikey'].dropna())
    property_full = set(properties['full_inchikey'].dropna())
    property_connectivity = set(properties['connectivity_inchikey'].dropna())

    exact_full = nmr_full & property_full
    shared_connectivity = nmr_connectivity & property_connectivity
    exact_connectivity = set(
        merged_index.loc[
            merged_index['full_inchikey'].isin(exact_full),
            'connectivity_inchikey',
        ].dropna()
    )

    matched_nmr_records = int(
        merged_index.loc[
            merged_index['full_inchikey'].isin(exact_full),
            'nmr_record_count',
        ].sum()
    )

    matched_properties = properties[
        properties['full_inchikey'].isin(exact_full)
    ].copy()
    details = matched_properties.merge(
        merged_index,
        on=['full_inchikey', 'connectivity_inchikey'],
        how='left',
    )
    details.insert(0, 'source', release['source'])
    details.insert(1, 'dataset', release['dataset'])
    details.insert(2, 'split', release['split'])
    details.insert(3, 'property_file', str(Path(release['path']).relative_to(REPO_ROOT)))

    summary = {
        'source': release['source'],
        'dataset': release['dataset'],
        'split': release['split'],
        'label_column': label_column,
        'property_rows': len(properties),
        'property_labeled_rows': int(properties[label_column].notna().sum()) if label_column else None,
        'property_invalid_or_unkeyed_rows': int(properties['full_inchikey'].isna().sum()),
        'property_unique_molecules': len(property_full),
        'property_rows_beyond_unique_molecules': int(properties['full_inchikey'].notna().sum() - len(property_full)),
        'exact_overlap_unique_molecules': len(exact_full),
        'exact_overlap_property_percent': round(100 * len(exact_full) / len(property_full), 2) if property_full else 0.0,
        'MERGED_DATASET_records_with_exact_overlap': matched_nmr_records,
        'connectivity_overlap_groups': len(shared_connectivity),
        'connectivity_only_groups_without_full_match': len(shared_connectivity - exact_connectivity),
    }
    return summary, details


releases = collect_property_releases()
reports = []
match_tables = []

for release in releases:
    print(f'Auditing {release["source"]}: {release["dataset"]} ({release["split"]})')
    report, details = audit_release(release, nmr_index)
    reports.append(report)
    match_tables.append(details)

overlap_summary = pd.DataFrame(reports).sort_values(
    ['source', 'split', 'exact_overlap_unique_molecules', 'dataset'],
    ascending=[True, True, False, True],
).reset_index(drop=True)

exact_matches = pd.concat(match_tables, ignore_index=True)

REPORT_ROOT.mkdir(parents=True, exist_ok=True)


def make_arrow_safe(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in result.select_dtypes(include='object').columns:
        result[column] = result[column].astype('string')
    return result


overlap_summary.to_csv(REPORT_ROOT / 'overlap_summary.csv', index=False)
overlap_summary.to_parquet(REPORT_ROOT / 'overlap_summary.parquet', index=False)
exact_matches.to_csv(REPORT_ROOT / 'exact_matches.csv', index=False)
make_arrow_safe(exact_matches).to_parquet(
    REPORT_ROOT / 'exact_matches.parquet',
    index=False,
)

tdc_test_summary = overlap_summary.loc[
    (overlap_summary['source'] == 'TDC ADMET')
    & (overlap_summary['split'] == 'test')
].sort_values('exact_overlap_unique_molecules', ascending=False)

FOCUSED_DATASETS = ['pampa_ncats', 'logd74_astrazeneca', 'sangster_logp']
focused_summary = overlap_summary.loc[
    overlap_summary['dataset'].isin(FOCUSED_DATASETS)
].sort_values('exact_overlap_unique_molecules', ascending=False)

focused_summary.to_csv(REPORT_ROOT / 'focused_pampa_logd_logp_overlap.csv', index=False)

print('\nRequested focused overlaps: PAMPA, logD7.4, Sangster logP')
print(focused_summary.to_string(index=False))
print('\nTDC ADMET: overlap with official test splits')
print(tdc_test_summary.to_string(index=False))
print('\nAll-source summary')
print(overlap_summary.to_string(index=False))
print(f'\nWrote reports to {REPORT_ROOT.relative_to(REPO_ROOT)}')


Auditing TDC ADMET: ames (test)
Auditing TDC ADMET: ames (train_val)
Auditing TDC ADMET: bbb_martins (test)
Auditing TDC ADMET: bbb_martins (train_val)
Auditing TDC ADMET: bioavailability_ma (test)
Auditing TDC ADMET: bioavailability_ma (train_val)
Auditing TDC ADMET: caco2_wang (test)
Auditing TDC ADMET: caco2_wang (train_val)
Auditing TDC ADMET: clearance_hepatocyte_az (test)
Auditing TDC ADMET: clearance_hepatocyte_az (train_val)
Auditing TDC ADMET: clearance_microsome_az (test)
Auditing TDC ADMET: clearance_microsome_az (train_val)
Auditing TDC ADMET: cyp2c9_substrate_carbonmangels (test)
Auditing TDC ADMET: cyp2c9_substrate_carbonmangels (train_val)
Auditing TDC ADMET: cyp2c9_veith (test)
Auditing TDC ADMET: cyp2c9_veith (train_val)
Auditing TDC ADMET: cyp2d6_substrate_carbonmangels (test)
Auditing TDC ADMET: cyp2d6_substrate_carbonmangels (train_val)
Auditing TDC ADMET: cyp2d6_veith (test)
Auditing TDC ADMET: cyp2d6_veith (train_val)
Auditing TDC ADMET: cyp3a4_substrate_carbonman

In [5]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


downloaded_files = [TDC_ARCHIVE]
downloaded_files.extend(
    MOLECULENET_ROOT / info['filename']
    for info in MOLECULENET_FILES.values()
)
downloaded_files.extend(
    TDC_EXTRA_ROOT / info['filename']
    for info in TDC_EXTRA_FILES.values()
)
downloaded_files.extend(
    SANGSTER_ROOT / info['filename']
    for info in SANGSTER_FILES.values()
)

manifest = {
    'MERGED_DATASET': {
        'path': str(MERGED_DATASET_PATH.relative_to(REPO_ROOT)),
        'identity_policy': 'RDKit full InChIKey primary; first InChIKey block reported only as a flagged connectivity fallback',
        **nmr_summary,
    },
    'sources': {
        'tdc_admet_url': TDC_ADMET_URL,
        'tdc_extra': TDC_EXTRA_FILES,
        'moleculenet': MOLECULENET_FILES,
        'sangster_logp': SANGSTER_FILES,
    },
    'downloaded_files': [
        {
            'path': str(path.relative_to(REPO_ROOT)),
            'size_bytes': path.stat().st_size,
            'sha256': sha256(path),
        }
        for path in downloaded_files
    ],
}

manifest_path = REPORT_ROOT / 'provenance.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f'Wrote {manifest_path.relative_to(REPO_ROOT)}')

# Exact matches are candidates suitable for a later, explicit molecule-level split.
# Do not use connectivity-only matches as labels.
exact_matches.head(20)


Wrote datasets_old/properties/admet_overlap/provenance.json


,source,dataset,split,property_file,Drug_ID,Drug,Y,_source_row,canonical_smiles_rdkit,full_inchikey,...,Ionisation-based class,logP,Adj pH,Temperature,Exp Method,Recommended,Lipinski descriptors,bRo5?,Source,_source_sheet
0,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 1191,O=c1c(=O)c2cccc3ccc4cccc1c4c32,0.0,3,O=c1c(=O)c2cccc3ccc4cccc1c4c32,JKCATVQPENLMQJ-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 7037,NC(=O)CN1CC(O)CC1=O,0.0,17,NC(=O)CN1CC(O)CC1=O,IHLAQQPQKRMGSS-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 452,CN1CCCC1=O,0.0,19,CN1CCCC1=O,SECXISVLQFMRJM-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 5814,C=CN1CCCC1=O,0.0,20,C=CN1CCCC1=O,WHNWPMSKXPGLAX-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 7586,CC(C)=C1CCC(C)CC1=O,0.0,25,CC(C)=C1CCC(C)CC1=O,NZGWDASTMWDZIW-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 6403,O=c1cc(CO)occ1O,1.0,37,O=c1cc(CO)occ1O,BEJNERDRQOWKJM-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 5815,Cc1occc(=O)c1O,1.0,39,Cc1occc(=O)c1O,XPCTZQVDEJYUGT-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 5969,c1ccc(-c2ccc(-c3ccccc3)cc2)cc1,0.0,43,c1ccc(-c2ccc(-c3ccccc3)cc2)cc1,XJKSTNDFUHDPQJ-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 436,Nc1ccc(-c2ccc(-c3ccc(N)cc3)cc2)cc1,1.0,45,Nc1ccc(-c2ccc(-c3ccc(N)cc3)cc2)cc1,QBSMHWVGUPQNJJ-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,TDC ADMET,ames,test,datasets_old/properties/tdc_admet/admet_group/...,Drug 5365,O=C1CCc2ccccc2O1,0.0,46,O=C1CCc2ccccc2O1,VMUXSMXIQBNMGZ-UHFFFAOYSA-N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
